## Imports and globals

In [ ]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_curve, auc
from sklearn.model_selection import learning_curve, GridSearchCV
from sklearn.decomposition import PCA
from skimage.feature import local_binary_pattern

# Plot styling
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'sans-serif',
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

DATASET_DIR = Path("dataset_splits_5_folds/fold_0")
IMG_SIZE = 256
RANDOM_SEED = 42

LBP_RADIUS = 3
LBP_POINTS = 8 * LBP_RADIUS

# Globals
trained_models = {}
eval_results = []

## Scaling

In [ ]:
def resize_with_padding(img, target_size=(256, 256)):
    if isinstance(target_size, int):
        target_size = (target_size, target_size)
        
    target_w, target_h = target_size
    h, w = img.shape
    
    scale = max(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    
    resized_img = cv2.resize(img, (new_w, new_h))
    
    start_x = (new_w - target_w) // 2
    start_y = (new_h - target_h) // 2
    
    final_img = resized_img[start_y:start_y+target_h, start_x:start_x+target_w]
    
    return final_img

In [ ]:
import random
import matplotlib.pyplot as plt

print("\n--- Generating Before/After Scaling Visualizations ---")
random.seed()
train_dir = DATASET_DIR / "train"
classes_on_disk = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])

sample_images = []
for class_name in classes_on_disk:
    img_list = list((train_dir / class_name).glob("*_orig.jpg"))
    if img_list:
        sample_images.append(random.choice(img_list))

fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(10, 12))
fig.suptitle("Image Resizing with Aspect Ratio Preservation (Cropping)", fontsize=16)

orig_shape = None

for i, img_path in enumerate(sample_images):
    orig_img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    orig_shape = orig_img.shape
    
    padded_img = resize_with_padding(orig_img, target_size=(256, 256))
    
    class_label = img_path.parent.name.capitalize()
    
    ax_orig = axes[i, 0]
    ax_orig.imshow(orig_img, cmap='gray')
    ax_orig.set_title(f"Original ({class_label})\nResolution: {orig_img.shape[1]}x{orig_img.shape[0]} px")
    ax_orig.axis('off')
    
    ax_padded = axes[i, 1]
    ax_padded.imshow(padded_img, cmap='gray')
    ax_padded.set_title(f"Cropped (1:1)\nResolution: {padded_img.shape[1]}x{padded_img.shape[0]} px")
    ax_padded.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("scaling_cropping_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

if orig_shape:
    print(f"Image data features before scaling (raw pixels): {orig_shape[0] * orig_shape[1]}")

print("Saved visualization as 'scaling_cropping_comparison.pdf'.")

# 1 Classic Machine Learning

## Extract and load data

In [ ]:
def extract_features(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
        
    img = resize_with_padding(img, IMG_SIZE)

    # Oloder resize
    #img = cv2.resize(img, IMG_SIZE)
    
    lbp = local_binary_pattern(img, LBP_POINTS, LBP_RADIUS, method="uniform")
    n_bins = LBP_POINTS + 2 
    hist, _ = np.histogram(lbp, density=True, bins=n_bins, range=(0, n_bins))
    
    mean_val = np.mean(img)
    std_val = np.std(img)
    return np.hstack([hist, mean_val, std_val])

def load_dataset(split_name):
    split_dir = DATASET_DIR / split_name
    features_list, labels_list = [], []
    class_names = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    
    for class_name in class_names:
        class_dir = split_dir / class_name
        image_paths = list(class_dir.glob("*.jpg"))
        for img_path in tqdm(image_paths, desc=f"Loading {split_name} - {class_name}"):
            features = extract_features(img_path)
            if features is not None:
                features_list.append(features)
                labels_list.append(class_name)
                
    return np.array(features_list), np.array(labels_list), class_names

print("--- Extracting LBP features ---")
X_train, y_train, classes = load_dataset("train")
X_test, y_test, _ = load_dataset("test")

# Label Encode
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_binarized = label_binarize(y_test, classes=classes)

print("--- Normalizing (Z-score) ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

print(f"Training data shape before LBP: {X_train.shape}")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Training data shape before LBP: {X_test.shape}")
print(f"Training data shape: {X_test_scaled.shape}")

## PCA 2D VIS

In [ ]:
print("--- Generating PCA Visualization (2D) ---")
pca_vis = PCA(n_components=2, random_state=RANDOM_SEED)
X_vis = pca_vis.fit_transform(X_train_scaled)

df_vis = pd.DataFrame({
    'PCA 1': X_vis[:, 0], 
    'PCA 2': X_vis[:, 1],
    'Class': y_train
})

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='PCA 1', y='PCA 2', 
    hue='Class', 
    palette="Set1", 
    data=df_vis, 
    alpha=0.7, 
    edgecolor="k"
)

plt.title("LBP Features in PCA Space (Training Set)")
plt.tight_layout()
plt.savefig("pca_2d_visualization.pdf", format='pdf', bbox_inches='tight')
plt.show()
print("Saved as pca_2d_visualization.pdf")

## PCA on LBP (95% var)

In [ ]:
print("--- Applying PCA on LBP features ---")
pca = PCA(n_components=0.95, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"Feature count after PCA (95% variance): {X_train_pca.shape[1]}")

## Only PCA on raw pixels (95% var)

In [ ]:
print("Preparing Naive Baseline (Raw Pixels + PCA)...")

def extract_raw_pixels(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
        
    img_small = resize_with_padding(img, (256, 256))
    return img_small.flatten()

X_train_naive, y_train_naive = [], []
X_test_naive, y_test_naive = [], []

for split_name, X_out, y_out in [("train", X_train_naive, y_train_naive), ("test", X_test_naive, y_test_naive)]:
    split_dir = DATASET_DIR / split_name
    for class_name in classes:
        for img_path in (split_dir / class_name).glob("*_orig.jpg"):
            features = extract_raw_pixels(img_path)
            if features is not None:
                X_out.append(features)
                y_out.append(class_name)

X_train_naive = np.array(X_train_naive)
X_test_naive = np.array(X_test_naive)

scaler_naive = StandardScaler()
pca_naive = PCA(n_components=0.95, random_state=RANDOM_SEED)

X_train_naive_pca = pca_naive.fit_transform(scaler_naive.fit_transform(X_train_naive))
X_test_naive_pca = pca_naive.transform(scaler_naive.transform(X_test_naive))

print(f"Features after Pure PCA (95% variance): {X_train_naive_pca.shape[1]}")

## Hiperparameters Opt (Grid Search)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
print("Starting hiperparameters opt (Grid Search)...")

param_grids = {
    "KNN": {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 2] # Istotne dla metryki minkowski
    },
    "RF": {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced', None] # Zabezpieczenie przed ewentualnym niezbalansowaniem
    },
    "SVM": {
        'C': [0.01, 0.1, 1, 10, 100, 1000],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
        'kernel': ['rbf', 'linear', 'poly'],
        'degree': [2, 3] # Używane tylko gdy kernel='poly'
    },
    "GBM": {
        'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
        'max_iter': [100, 200, 300, 500],
        'max_depth': [None, 3, 5, 10],
        'l2_regularization': [0.0, 0.1, 1.0, 10.0]
    },
    "MLP": {
        'hidden_layer_sizes': [(64,), (128,), (256,), (128, 64), (256, 128), (256, 128, 64)],
        'alpha': [0.0001, 0.001, 0.01, 0.1],
        'learning_rate_init': [0.001, 0.01],
        'activation': ['relu', 'tanh']
    }
}

base_models = {
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "RF": RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    "SVM": SVC(probability=True, random_state=RANDOM_SEED),
    "GBM": HistGradientBoostingClassifier(random_state=RANDOM_SEED),
    "MLP": MLPClassifier(max_iter=2000, random_state=RANDOM_SEED, early_stopping=True)
}

best_estimators = {}
all_cv_results = {}

for name, model in base_models.items():
    print(f"Searching best architecture for: {name}...")
    gs = GridSearchCV(model, param_grids[name], cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    gs.fit(X_train_scaled, y_train_encoded)
    
    best_estimators[name] = gs.best_estimator_
    all_cv_results[name] = pd.DataFrame(gs.cv_results_)
    
    print(f"Optimal parameters for {name}: {gs.best_params_}\n")
    
    all_cv_results[name].to_csv(f"{name}_gridsearch_results.csv", index=False)

print("Optimization completed. Proceeding to final model training.")

## Training base machine learning models

In [ ]:
from sklearn.base import clone

model_variants = {
    "KNN (LBP)": (best_estimators["KNN"], X_train_scaled, X_test_scaled),
    "KNN (LBP+PCA)": (best_estimators["KNN"], X_train_pca, X_test_pca),
    "KNN (Pure PCA)": (best_estimators["KNN"], X_train_naive_pca, X_test_naive_pca),
    "RF (LBP)": (best_estimators["RF"], X_train_scaled, X_test_scaled),
    "RF (LBP+PCA)": (best_estimators["RF"], X_train_pca, X_test_pca),
    "RF (Pure PCA)": (best_estimators["RF"], X_train_naive_pca, X_test_naive_pca),
    "SVM (LBP)": (best_estimators["SVM"], X_train_scaled, X_test_scaled),
    "SVM (LBP+PCA)": (best_estimators["SVM"], X_train_pca, X_test_pca),
    "SVM (Pure PCA)": (best_estimators["SVM"], X_train_naive_pca, X_test_naive_pca),
    "GBM (LBP)": (best_estimators["GBM"], X_train_scaled, X_test_scaled),
    "GBM (LBP+PCA)": (best_estimators["GBM"], X_train_pca, X_test_pca),
    "GBM (Pure PCA)": (best_estimators["GBM"], X_train_naive_pca, X_test_naive_pca),
    "MLP (LBP)": (best_estimators["MLP"], X_train_scaled, X_test_scaled),
    "MLP (LBP+PCA)": (best_estimators["MLP"], X_train_pca, X_test_pca),
    "MLP (Pure PCA)": (best_estimators["MLP"], X_train_naive_pca, X_test_naive_pca)
}

for name, (base_model, X_tr, X_te) in model_variants.items():
    print(f"Training: {name}...")

    model = clone(base_model)
    
    start_train = time.time()
    if "Pure PCA" in name:
        current_y_train = le.transform(y_train_naive)
        current_y_test = y_test_naive
    else:
        current_y_train = y_train_encoded
        current_y_test = y_test
    
    model.fit(X_tr, current_y_train)
    train_time = time.time() - start_train
    
    start_test = time.time()
    y_pred_encoded = model.predict(X_te)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_te)
    else:
        y_proba = None
    test_time = time.time() - start_test
    
    y_pred = le.inverse_transform(y_pred_encoded)
    acc = accuracy_score(current_y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(current_y_test, y_pred, average='macro', zero_division=0)
    
    eval_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "F1-Score": round(f1, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "Train Time (s)": round(train_time, 2),
        "Test Time (s)": round(test_time, 4)
    })
    
    trained_models[name] = {"model": model, "y_pred": y_pred, "y_proba": y_proba, "X_tr": X_tr, "y_true": current_y_test}

print("\nFinished.")

## Experiment n=5 different folds and mean+std

In [ ]:
import cv2
import numpy as np
import pandas as pd
import time
from pathlib import Path
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def extract_lbp_features(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = resize_with_padding(img, IMG_SIZE)
    lbp = local_binary_pattern(img, LBP_POINTS, LBP_RADIUS, method="uniform")
    n_bins = LBP_POINTS + 2 
    hist, _ = np.histogram(lbp, density=True, bins=n_bins, range=(0, n_bins))
    return np.hstack([hist, np.mean(img), np.std(img)])

def extract_raw_pixels(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img_small = resize_with_padding(img, (256, 256))
    return img_small.flatten()

def load_features_from_dir(directory_path, extract_func, desc_label, suffix="*.jpg"):
    features_list, labels_list = [], []
    class_names = sorted([d.name for d in directory_path.iterdir() if d.is_dir()])
    for class_name in class_names:
        print(f" -> Loading: {desc_label} | Klasa: {class_name}")
        class_dir = directory_path / class_name
        image_paths = list(class_dir.glob(suffix))
        for img_path in image_paths:
            features = extract_func(img_path)
            if features is not None:
                features_list.append(features)
                labels_list.append(class_name) 
    return np.array(features_list), np.array(labels_list), class_names

BASE_DIR = Path("dataset_splits_5_folds")
classes = sorted([d.name for d in (BASE_DIR / "fold_0" / "train").iterdir() if d.is_dir()])

le = LabelEncoder()
le.fit(classes)

classical_eval_results = []
trained_models_classical = {}

print("Starting 5-Fold Cross Validation for machine learning methods...\n")

for fold_idx in range(5):
    print(f"\n{'='*50}")
    print(f"--- Proccessing {fold_idx} ---")
    print(f"{'='*50}")
    
    train_dir = BASE_DIR / f"fold_{fold_idx}" / "train"
    test_dir = BASE_DIR / f"fold_{fold_idx}" / "test"
    
    print("1. Features extraction (Train/Test)...")
    X_train_lbp, y_train_lbp, _ = load_features_from_dir(train_dir, extract_lbp_features, "LBP Train")
    X_test_lbp, y_test_lbp, _ = load_features_from_dir(test_dir, extract_lbp_features, "LBP Test")
    
    X_train_naive, y_train_naive, _ = load_features_from_dir(train_dir, extract_raw_pixels, "Naive Train", suffix="*_orig.jpg")
    X_test_naive, y_test_naive, _ = load_features_from_dir(test_dir, extract_raw_pixels, "Naive Test", suffix="*_orig.jpg")
    
    y_train_enc = le.transform(y_train_lbp)
    y_test_enc = le.transform(y_test_lbp)
    y_train_naive_enc = le.transform(y_train_naive)
    y_test_naive_enc = le.transform(y_test_naive)

    print("2. Scale and featture reduction (PCA)...")
    scaler_lbp = StandardScaler()
    X_train_lbp_scaled = scaler_lbp.fit_transform(X_train_lbp)
    X_test_lbp_scaled = scaler_lbp.transform(X_test_lbp)
    
    pca_lbp = PCA(n_components=0.95, random_state=RANDOM_SEED)
    X_train_lbp_pca = pca_lbp.fit_transform(X_train_lbp_scaled)
    X_test_lbp_pca = pca_lbp.transform(X_test_lbp_scaled)
    
    scaler_naive = StandardScaler()
    X_train_naive_scaled = scaler_naive.fit_transform(X_train_naive)
    X_test_naive_scaled = scaler_naive.transform(X_test_naive)
    
    pca_naive = PCA(n_components=0.95, random_state=RANDOM_SEED)
    X_train_naive_pca = pca_naive.fit_transform(X_train_naive_scaled)
    X_test_naive_pca = pca_naive.transform(X_test_naive_scaled)

    print("3. Training and eval...")
    model_variants = {
        "KNN (LBP)": (best_estimators["KNN"], X_train_lbp_scaled, X_test_lbp_scaled, y_train_enc, y_test_enc),
        "KNN (LBP+PCA)": (best_estimators["KNN"], X_train_lbp_pca, X_test_lbp_pca, y_train_enc, y_test_enc),
        "KNN (Pure PCA)": (best_estimators["KNN"], X_train_naive_pca, X_test_naive_pca, y_train_naive_enc, y_test_naive_enc),
        "RF (LBP)": (best_estimators["RF"], X_train_lbp_scaled, X_test_lbp_scaled, y_train_enc, y_test_enc),
        "RF (LBP+PCA)": (best_estimators["RF"], X_train_lbp_pca, X_test_lbp_pca, y_train_enc, y_test_enc),
        "RF (Pure PCA)": (best_estimators["RF"], X_train_naive_pca, X_test_naive_pca, y_train_naive_enc, y_test_naive_enc),
        "SVM (LBP)": (best_estimators["SVM"], X_train_lbp_scaled, X_test_lbp_scaled, y_train_enc, y_test_enc),
        "SVM (LBP+PCA)": (best_estimators["SVM"], X_train_lbp_pca, X_test_lbp_pca, y_train_enc, y_test_enc),
        "SVM (Pure PCA)": (best_estimators["SVM"], X_train_naive_pca, X_test_naive_pca, y_train_naive_enc, y_test_naive_enc),
        "GBM (LBP)": (best_estimators["GBM"], X_train_lbp_scaled, X_test_lbp_scaled, y_train_enc, y_test_enc),
        "GBM (LBP+PCA)": (best_estimators["GBM"], X_train_lbp_pca, X_test_lbp_pca, y_train_enc, y_test_enc),
        "GBM (Pure PCA)": (best_estimators["GBM"], X_train_naive_pca, X_test_naive_pca, y_train_naive_enc, y_test_naive_enc),
        "MLP (LBP)": (best_estimators["MLP"], X_train_lbp_scaled, X_test_lbp_scaled, y_train_enc, y_test_enc),
        "MLP (LBP+PCA)": (best_estimators["MLP"], X_train_lbp_pca, X_test_lbp_pca, y_train_enc, y_test_enc),
        "MLP (Pure PCA)": (best_estimators["MLP"], X_train_naive_pca, X_test_naive_pca, y_train_naive_enc, y_test_naive_enc)
    }

    for name, (base_model, X_tr, X_te, curr_y_train, curr_y_test) in model_variants.items():
        model = clone(base_model)
        
        start_train = time.time()
        model.fit(X_tr, curr_y_train)
        train_time = time.time() - start_train
        
        start_test = time.time()
        y_pred_encoded = model.predict(X_te)
        test_time = time.time() - start_test
        
        acc = accuracy_score(curr_y_test, y_pred_encoded)
        prec, rec, f1, _ = precision_recall_fscore_support(curr_y_test, y_pred_encoded, average='macro', zero_division=0)
        
        classical_eval_results.append({
            "Fold": fold_idx,
            "Model": name,
            "Accuracy": round(acc, 4),
            "F1-Score": round(f1, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "Train Time (s)": round(train_time, 2),
            "Test Time (s)": round(test_time, 4)
        })

print("\n--- FINISHED TRAINING ---")

df_classical_metrics = pd.DataFrame(classical_eval_results)

final_classical_df = df_classical_metrics.groupby("Model").agg({
    "Accuracy": ["mean", "std"],
    "Precision": ["mean", "std"],
    "Recall": ["mean", "std"],
    "F1-Score": ["mean", "std"],
    "Train Time (s)": ["mean"]
}).round(4)

print("\nRESULTS (Mean 5 Folds):")
display(final_classical_df)

df_classical_metrics.to_csv("results_mean_classic_5fold.csv", index=False)
final_classical_df.to_csv("results_mean_classic_5fold.csv")

with open("classical_results_5folds.tex", "w") as f:
    f.write(final_classical_df.reset_index().to_latex(index=False, caption="Evaluation Metrics for Classical ML Models (5-Fold CV)"))

## Visualisation

In [ ]:
df_results = pd.DataFrame(eval_results)
display(df_results)
df_results.to_csv("classical_models_results.csv", index=False)

# Export to LaTeX
latex_table = df_results.to_latex(index=False, caption="Evaluation Metrics for Classical Machine Learning Models", label="tab:classical_results")
with open("classical_models_results.tex", "w") as f:
    f.write(latex_table)

print("\n--- Generating Plots ---")
for name, data in tqdm(trained_models.items(), desc="Creating Visualizations"):
    safe_name = "".join([c if c.isalnum() else "_" for c in name])
    y_true = data["y_true"]
    y_pred = data["y_pred"]
    y_proba = data["y_proba"]
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=classes, normalize='true')
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'cm_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    
    # ROC Curves (only for LBP without PCA)
    if "LBP" in name and "PCA" not in name and y_proba is not None: 
        plt.figure(figsize=(8, 6))
        for i, class_name in enumerate(classes):
            fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_proba[:, i])
            plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {auc(fpr, tpr):.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.title(f'ROC Curves - {name}')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f'roc_{safe_name}.pdf', format='pdf', bbox_inches='tight')
        plt.close()

# Learning Curve for best MLP
best_mlp = trained_models["MLP (LBP)"]["model"]
if hasattr(best_mlp, 'loss_curve_') and hasattr(best_mlp, 'validation_scores_'):
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(best_mlp.loss_curve_, color='blue', label='Training Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    
    ax2 = ax1.twinx()
    ax2.plot(best_mlp.validation_scores_, color='green', label='Validation Accuracy')
    ax2.set_ylabel('Accuracy', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    
    plt.title('Learning Curve - MLP Classifier (Early Stopping)')
    fig.tight_layout()
    plt.savefig('mlp_loss_curve.pdf', format='pdf', bbox_inches='tight')
    

print("Evaluation complete. All plots and LaTeX tables saved.")

# Deep Learning - Convolutional Networks

## Imports, consts and data load

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import time
import numpy as np
import copy
import pandas as pd
from PIL import Image

print("CUDA Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

DATASET_DIR = Path("dataset_splits_5_folds/fold_0")
BATCH_SIZE = 64
EPOCHS = 300
PATIENCE = 20 # Early stopping
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

mean_imagenet = [0.485, 0.456, 0.406]
std_imagenet = [0.229, 0.224, 0.225]

# Preserving aspect ratio + Center Crop
data_transforms = {
    x: transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean_imagenet, std_imagenet)
    ]) for x in ['train', 'val', 'test']
}

image_datasets = {x: datasets.ImageFolder(DATASET_DIR / x, data_transforms[x]) for x in ['train', 'val', 'test']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=8, pin_memory=True) for x in ['train', 'val', 'test']}

class_names = image_datasets['train'].classes
num_classes = len(class_names)
print(f"\nFound {num_classes} classes: {class_names}")
print(f"Dataset Size -> Train: {len(image_datasets['train'])}, Val: {len(image_datasets['val'])}, Test: {len(image_datasets['test'])}")

dl_results = []
trained_dl_models = {}

In [ ]:
import torchvision

print("\n--- Visualizing a Batch of Transformed Images (PyTorch Input) ---")

inputs, labels = next(iter(dataloaders['train']))

num_images = 4
inputs_to_show = inputs[:num_images]
labels_to_show = labels[:num_images]

def imshow_batch(inp, title=None):
    inp = inp.numpy().transpose((1, 2, 0)) 
    
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    
    plt.figure(figsize=(12, 4))
    plt.imshow(inp)
    if title is not None:
        plt.title(title, fontsize=14, pad=15)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig("pytorch_dataloader_check.pdf", format='pdf', bbox_inches='tight')
    plt.show()

out_grid = torchvision.utils.make_grid(inputs_to_show, nrow=4)

class_titles = [class_names[x].capitalize() for x in labels_to_show]
title_str = " | ".join(class_titles) + f"\nTensor Shape per Image: [Channels: {inputs[0].size(0)}, Height: {inputs[0].size(1)}, Width: {inputs[0].size(2)}]"

imshow_batch(out_grid, title=title_str)

print(f"Global Batch Shape: {inputs.shape} -> (Batch Size, Channels, Height, Width)")
print("Saved PyTorch input visualization as 'pytorch_dataloader_check.pdf'.")

## Architectures definitions

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.adapt_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.adapt_pool(x)
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def get_resnet18(num_classes, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

def get_alexnet(num_classes, pretrained=True):
    weights = models.AlexNet_Weights.DEFAULT if pretrained else None
    model = models.alexnet(weights=weights)
    model.classifier[6] = nn.Linear(4096, num_classes)
    return model

def get_densenet121(num_classes, pretrained=True):
    weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
    model = models.densenet121(weights=weights)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, num_classes)
    return model

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

### Proposed upgrade to CustomCNN ( original CustomCNN worked better for scaling images without preserving proportions, CustomCNN_V2 network works much better for cropped photos than V1)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN_V2(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN_V2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.adapt_pool = nn.AdaptiveAvgPool2d((7, 7))
        
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        
        x = self.adapt_pool(x)
        x = x.view(x.size(0), -1) 
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## Train and eval functions

In [ ]:
def train_model(model, model_name, criterion, optimizer, num_epochs=EPOCHS):
    print(f"\n{'='*50}\nTraining Started: {model_name}\n{'='*50}")
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    early_stopper = EarlyStopping(patience=PATIENCE)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}', end=' ')
        
        for phase in ['train', 'val']:
            if phase == 'train': 
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            print(f'| {phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}', end=' ')

            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                early_stopper(epoch_loss)

        print()
        if early_stopper.early_stop:
            print(f"No improvement for {PATIENCE} epochs. Early Stopping triggered at epoch {epoch+1}!")
            break

    time_elapsed = time.time() - since
    print(f'\nTraining completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    
    safe_name = "".join([c if c.isalnum() else "_" for c in model_name])
    torch.save(model.state_dict(), f"weights_{safe_name}.pth")
    print(f"Saved model weights as weights_{safe_name}.pth")
    
    plt.figure(figsize=(6, 5))
    plt.plot(history['train_loss'], color='tab:blue', linestyle='-', linewidth=2, label='Training Loss')
    plt.plot(history['val_loss'], color='tab:orange', linestyle='--', linewidth=2, label='Validation Loss')
    plt.xlabel('Epochs', fontsize=11)
    plt.ylabel('Loss', fontsize=11)
    plt.title(f'Loss Curve - {model_name}', fontsize=12, pad=15)
    plt.legend(frameon=True, loc='best')
    plt.tight_layout()
    plt.savefig(f'dl_loss_curve_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    
    plt.figure(figsize=(6, 5))
    plt.plot(history['train_acc'], color='tab:blue', linestyle='-', linewidth=2, label='Training Accuracy')
    plt.plot(history['val_acc'], color='tab:orange', linestyle='--', linewidth=2, label='Validation Accuracy')
    plt.xlabel('Epochs', fontsize=11)
    plt.ylabel('Accuracy', fontsize=11)
    plt.title(f'Accuracy Curve - {model_name}', fontsize=12, pad=15)
    plt.legend(frameon=True, loc='best')
    plt.tight_layout()
    plt.savefig(f'dl_acc_curve_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    
    return model

def evaluate_dl_model(model, model_name):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for inputs, labels in dataloaders['test']:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    all_probs, all_preds, all_labels = np.array(all_probs), np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    
    trained_dl_models[model_name] = {
        "model": model, "probs": all_probs, "preds": all_preds, "labels": all_labels
    }
    
    return {"Model": model_name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}

## Calculate params

In [ ]:
dl_variants = {
    "Custom CNN V2 (From Scratch)": CustomCNN_V2(num_classes).to(DEVICE),
    "AlexNet (From Scratch)": get_alexnet(num_classes, pretrained=False).to(DEVICE),
    "AlexNet (Transfer Learning)": get_alexnet(num_classes, pretrained=True).to(DEVICE),
    "ResNet18 (From Scratch)": get_resnet18(num_classes, pretrained=False).to(DEVICE),
    "ResNet18 (Transfer Learning)": get_resnet18(num_classes, pretrained=True).to(DEVICE),
    "DenseNet121 (From Scratch)": get_densenet121(num_classes, pretrained=False).to(DEVICE),
    "DenseNet121 (Transfer Learning)": get_densenet121(num_classes, pretrained=True).to(DEVICE)
}

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

print(f"\n{'='*75}")
print(f"{'Model Variant':<35} | {'Total Params':<15} | {'Trainable Params':<15}")
print(f"{'-'*75}")

for name, net in dl_variants.items():
    total, trainable = count_parameters(net)
    print(f"{name:<35} | {total:<15,d} | {trainable:<15,d}")
    
print(f"{'='*75}\n")

## All variants training - no repetition/no mean

In [ ]:
criterion = nn.CrossEntropyLoss()

dl_variants = {
    "Custom CNN V2 (From Scratch)": CustomCNN_V2(num_classes).to(DEVICE),
    "AlexNet (From Scratch)": get_alexnet(num_classes, pretrained=False).to(DEVICE),
    "AlexNet (Transfer Learning)": get_alexnet(num_classes, pretrained=True).to(DEVICE),
    "ResNet18 (From Scratch)": get_resnet18(num_classes, pretrained=False).to(DEVICE),
    "ResNet18 (Transfer Learning)": get_resnet18(num_classes, pretrained=True).to(DEVICE),
    "DenseNet121 (From Scratch)": get_densenet121(num_classes, pretrained=False).to(DEVICE),
    "DenseNet121 (Transfer Learning)": get_densenet121(num_classes, pretrained=True).to(DEVICE)
}

for name, net in dl_variants.items():
    optimizer = optim.Adam(net.parameters(), lr=0.0001)
    trained_net = train_model(net, name, criterion, optimizer)
    metrics = evaluate_dl_model(trained_net, name)
    dl_results.append(metrics)

## 5 Folds with means and std

In [ ]:
import pandas as pd

criterion = nn.CrossEntropyLoss()
BASE_DIR = Path("dataset_splits_5_folds")

all_folds_metrics = {}
all_folds_trained_models = {}

for fold_idx in range(5):
    print(f"\n{'='*60}")
    print(f"STARTING FOLD {fold_idx}")
    print(f"{'='*60}")
    
    current_fold_dir = BASE_DIR / f"fold_{fold_idx}"
    image_datasets = {x: datasets.ImageFolder(current_fold_dir / x, data_transforms[x]) for x in ['train', 'val', 'test']}
    dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train')) for x in ['train', 'val', 'test']}
    
    dl_variants = {
        "Custom CNN V2 (From Scratch)": CustomCNN_V2(num_classes).to(DEVICE),
        "AlexNet (From Scratch)": get_alexnet(num_classes, pretrained=False).to(DEVICE),
        "AlexNet (Transfer Learning)": get_alexnet(num_classes, pretrained=True).to(DEVICE),
        "ResNet18 (From Scratch)": get_resnet18(num_classes, pretrained=False).to(DEVICE),
        "ResNet18 (Transfer Learning)": get_resnet18(num_classes, pretrained=True).to(DEVICE),
        "DenseNet121 (From Scratch)": get_densenet121(num_classes, pretrained=False).to(DEVICE),
        "DenseNet121 (Transfer Learning)": get_densenet121(num_classes, pretrained=True).to(DEVICE)
    }

    dl_results = []
    trained_dl_models = {}

    for name, net in dl_variants.items():
        optimizer = optim.Adam(net.parameters(), lr=0.0001)
        
        fold_model_name = f"{name}_Fold_{fold_idx}"
        trained_net = train_model(net, fold_model_name, criterion, optimizer)
        
        metrics = evaluate_dl_model(trained_net, name)
        
        metrics["Fold"] = fold_idx
        dl_results.append(metrics)

    fold_key = f"Fold_{fold_idx}"
    all_folds_metrics[fold_key] = dl_results
    all_folds_trained_models[fold_key] = trained_dl_models.copy() 


flat_metrics = [metric for fold_metrics in all_folds_metrics.values() for metric in fold_metrics]
df_metrics = pd.DataFrame(flat_metrics)

final_results_df = df_metrics.groupby("Model").agg({
    "Accuracy": ["mean", "std"],
    "Precision": ["mean", "std"],
    "Recall": ["mean", "std"],
    "F1-Score": ["mean", "std"]
}).round(4)

print("\n\n" + "*"*50)
print("FINAL RESULTS (Mean from 5 folds)")
print("*"*50)
display(final_results_df)
final_results_df.to_csv("results_5folds_dl.csv")

latex_dl_folds = final_results_df.reset_index().to_latex(
    index=False, 
    caption="Evaluation Metrics for Deep Learning Architectures", 
    label="tab:dl_results"
)

with open("deep_learning_results_5folds.tex", "w") as f:
    f.write(latex_dl_folds)

print("\nSet fold0 for confusion matrices and ROC...")
dl_results = all_folds_metrics["Fold_0"]
trained_dl_models = all_folds_trained_models["Fold_0"]

## Generate confusion matrices and ROC curves for all models

In [ ]:
df_dl_results = pd.DataFrame(dl_results)
display(df_dl_results)
df_dl_results.to_csv("deep_learning_results.csv", index=False)

# Export to LaTeX
latex_dl = df_dl_results.to_latex(index=False, caption="Evaluation Metrics for Deep Learning Architectures", label="tab:dl_results")
with open("deep_learning_results.tex", "w") as f:
    f.write(latex_dl)

print("\nGenerating Deep Learning Visualizations...")
for name, data in tqdm(trained_dl_models.items(), desc="Creating Matrices and ROC Curves"):
    safe_name = "".join([c if c.isalnum() else "_" for c in name])
    
    # Confusion Matrix
    cm = confusion_matrix(data["labels"], data["preds"], normalize="true")
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'dl_cm_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    
    # ROC Curves
    y_test_bin = label_binarize(data["labels"], classes=range(num_classes))
    plt.figure(figsize=(8, 6))
    for i, cls_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], data["probs"][:, i])
        plt.plot(fpr, tpr, lw=2, label=f'{cls_name.capitalize()} (AUC = {auc(fpr, tpr):.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.title(f'ROC Curves - {name}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f'dl_roc_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()

print("Deep Learning visualisations generated successfully!")

## XAI Grad-CAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Generating Grad-CAM Activation Maps...")
inputs, labels = next(iter(dataloaders['test']))
inputs = inputs.to(DEVICE)

images_per_class = {}

for batch_inputs, batch_labels in dataloaders['test']:
    for i in range(batch_inputs.size(0)):
        label_idx = batch_labels[i].item()
        
        if label_idx not in images_per_class:
            images_per_class[label_idx] = (batch_inputs[i], label_idx)
            
        if len(images_per_class) == len(class_names):
            break
    if len(images_per_class) == len(class_names):
        break

selected_images = [images_per_class[k][0] for k in sorted(images_per_class.keys())]
selected_labels = [images_per_class[k][1] for k in sorted(images_per_class.keys())]

inputs = torch.stack(selected_images).to(DEVICE)
labels = torch.tensor(selected_labels)
num_images_to_show = len(class_names)

def denormalize(tensor):
    mean = np.array(mean_imagenet).reshape(1, 1, 3)
    std = np.array(std_imagenet).reshape(1, 1, 3)
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = np.clip(std * img + mean, 0, 1)
    return img


for name, model_data in trained_dl_models.items():
    model = model_data["model"]
    model.eval()
    
    if "Custom CNN" in name: target_layers = [model.conv2]
    elif "Custom CNN V2" in name: target_layers = [model.conv3]
    elif "ResNet" in name: target_layers = [model.layer4[-1]]
    elif "AlexNet" in name: target_layers = [model.features[-1]]
    elif "DenseNet" in name: target_layers = [model.features[-1]]
    else: continue
        
    cam = GradCAM(model=model, target_layers=target_layers)
    fig, axes = plt.subplots(num_images_to_show, 3, figsize=(12, 4 * num_images_to_show))
    fig.suptitle(f"Grad-CAM Decision Process: {name}", fontsize=16, y=1.02)
    
    for i in range(num_images_to_show):
        input_tensor = inputs[i].unsqueeze(0) 
        true_label = class_names[labels[i].item()].capitalize()
        
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
        rgb_img = denormalize(inputs[i])
        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        
        with torch.no_grad():
            pred_idx = torch.max(model(input_tensor), 1)[1].item()
            pred_class = class_names[pred_idx].capitalize()

        axes[i, 0].imshow(rgb_img); axes[i, 0].set_title(f"Original ({true_label})"); axes[i, 0].axis('off')
        axes[i, 1].imshow(grayscale_cam, cmap='jet'); axes[i, 1].set_title("Activation Heatmap"); axes[i, 1].axis('off')
        axes[i, 2].imshow(cam_image); color = "green" if labels[i].item() == pred_idx else "red"
        axes[i, 2].set_title(f"Prediction ({pred_class})", color=color); axes[i, 2].axis('off')

    plt.tight_layout()
    safe_name = "".join([c if c.isalnum() else "_" for c in name])
    plt.savefig(f'xai_gradcam_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print(f"Generated Grad-CAM for: {name}")